# Chạy thử nghiệm E09.15 trên Google Colab (`duytung/01-cub-gom-cum-cosine-gmm`)
Notebook này tự động tải `CUB-200` từ HuggingFace, chạy Phase 1 (`r0`), tạo snapshot `colab_cub_ft_l25_r0_s0..s4`, rồi chạy so sánh `FTR1` vs `FTR1_cos` vs `FTR1_cos_z` vs `FTR1_gmm`.

In [ ]:
# 1. Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone / cập nhật repo public và dọn file tạm (nếu có) để git sạch
import os
WORK_DIR = '/content/drive/MyDrive/VB_CGCD_Work'
REPO_DIR = f'{WORK_DIR}/VB-CGCD-defect-testing-only'
os.makedirs(WORK_DIR, exist_ok=True)

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/DuytungDeveloper/VB-CGCD-defect-testing-only.git {REPO_DIR}

%cd {REPO_DIR}
!git checkout -- notes/LOG.md runs/INDEX.txt 2>/dev/null || true
!rm -rf runs/20261122_* 2>/dev/null || true
!git fetch origin
!git checkout duytung/01-cub-gom-cum-cosine-gmm
!git pull origin duytung/01-cub-gom-cum-cosine-gmm
!git status -sb

In [ ]:
# 3. Kiểm tra GPU & thư viện có sẵn trên Colab
!python -c "import torch, jax, sklearn, scipy, datasets; print('Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available(), '| JAX:', jax.__version__)"

In [ ]:
# 4. Tự động tải CUB-200 từ HuggingFace, chạy Phase 1 (r0) & tạo 5 chuỗi s0..s4 nếu chưa có
import glob, subprocess

feat_dirs = sorted(glob.glob('features/*__colab_cub_ft_l25_r0/dino-vitb16-sl'))
if not feat_dirs:
    print('>>> Đang tải CUB-200 từ HuggingFace & chạy Phase 1 (FT seed r0, ~15 phút trên T4)...')
    !FT_SEED=0 FT_EPOCHS=10 FT_BATCH_SIZE=32 FT_ACCUM=4 FT_LABELED_IDX=splits/cub_labeled_hf_idx.npy \
      python scripts/exp.py run colab_cub_ft_l25_r0 --inputs - -- bash -c \
      'O="features/$(basename "$EXP_RUN_DIR")"; mkdir -p "$O" && FT_CKPT_PATH="$O/ft_ckpt.pt" python -u feature_extractor/dino-cub200.py --finetuned --output_dir "$O" --batch-size 128 --seed "$FT_SEED"'
    feat_dirs = sorted(glob.glob('features/*__colab_cub_ft_l25_r0/dino-vitb16-sl'))
FEAT_DIR = feat_dirs[-1]
print('FEAT_DIR =', FEAT_DIR)

for s in range(5):
    tag = f'colab_cub_ft_l25_r0_s{s}'
    if not glob.glob(f'runs/*__{tag}/snap/04.pkl'):
        print(f'>>> Tạo chuỗi {tag}...')
        !OMP_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 MKL_NUM_THREADS=1 \
          python scripts/exp.py run {tag} --inputs {FEAT_DIR} -- python lab/sessions.py run --features {FEAT_DIR} --seed {s} --dataset cub200

# Commit các log vừa tạo vào nhánh local để exp.py cho phép chạy bước tiếp theo
!git config user.name 'Colab Runner'
!git config user.email 'colab@example.com'
!git add notes/LOG.md runs/INDEX.txt runs/*__colab_cub_ft_l25*
!git diff --cached --quiet || git commit -m "colab: luu log colab_cub_ft_l25_r0"

In [ ]:
# 5. Chạy kiểm tra nhanh (--checks-only)
!OMP_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 MKL_NUM_THREADS=1 \
  python lab/score.py e0915 --prefix colab_cub_ft_l25 --fts 0 --checks-only

In [ ]:
# 6. Chạy chính thức E09.15 trên FT seed r0 (5 chuỗi s0..s4)
!OMP_NUM_THREADS=1 OPENBLAS_NUM_THREADS=1 MKL_NUM_THREADS=1 \
  python scripts/exp.py run colab_e09_15_ft_l25 -- python lab/score.py e0915 --prefix colab_cub_ft_l25 --fts 0

In [ ]:
# 7. In bảng kết quả tổng hợp
!python lab/score.py e0915-summary